# 02. Baseline

Melatih Logistic Regression dan Random Forest (seed 0) dengan `make_model` dari `src/run_experiments.py`, lalu mengevaluasi baseline threshold 0,5 pada data uji dengan `compute_metrics` dari `src/evaluation.py`. Hasilnya dicocokkan dengan `results/metrics.csv`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent          # notebook dijalankan dari folder notebooks/
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "KaggleV2-May-2016.csv"
if not DATA.exists():
    raise FileNotFoundError(f"Taruh CSV Kaggle di {DATA} (lihat data/README.md).")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

In [2]:
from preprocessing import load_raw, clean, chronological_split, feature_columns
from evaluation import compute_metrics
from run_experiments import make_model, describe_features

df, report = clean(load_raw(str(DATA)))
train, val, test = chronological_split(df)
feats = feature_columns(exclude_sms=False)
y_tr, y_te = train["target"].values, test["target"].values
print(f"latih {len(train)} | validasi {len(val)} | uji {len(test)}")

latih 67360 | validasi 21175 | uji 21986


In [3]:
models, scores = {}, {}
for name in ["logreg", "random_forest"]:
    pipe = make_model(name, seed=0, exclude_sms=False).fit(train[feats], y_tr)
    models[name] = pipe
    scores[name] = pipe.predict_proba(test[feats])[:, 1]
print(describe_features(models["logreg"].named_steps["prep"]))

{'n_features_total': 87, 'n_numeric_scaled': 6, 'numeric_scaled_columns': ['age', 'lead_days', 'scheduled_hour', 'appt_weekday', 'sched_weekday', 'handicap'], 'n_binary_passthrough': 6, 'neighbourhood_categories_in_train': 81, 'neighbourhood_infrequent_merged': 7, 'neighbourhood_onehot_columns': 75, 'class_weight': 'None (default scikit-learn; tanpa resampling)'}


## Baseline threshold 0,5 dan pembanding tanpa model

In [4]:
rows = {}
for name, s in scores.items():
    rows[name] = compute_metrics(y_te, s, threshold=0.5, seed=0)
rows["lead_days (heuristik)"] = compute_metrics(y_te, test["lead_days"].values.astype(float), seed=0, probabilistic=False)
rows["acak (seed 0)"] = compute_metrics(y_te, np.random.default_rng(0).random(len(y_te)), seed=0, probabilistic=False)
table = pd.DataFrame(rows).T
table[["pr_auc", "roc_auc", "precision_at_10pct", "recall_at_10pct", "lift_at_10pct", "precision", "recall", "flagged_fraction"]].round(3)

,pr_auc,roc_auc,precision_at_10pct,recall_at_10pct,lift_at_10pct,precision,recall,flagged_fraction
logreg,0.295,0.663,0.339,0.183,1.833,0.411,0.041,0.019
random_forest,0.325,0.725,0.370,0.200,2.000,0.243,0.002,0.002
lead_days (heuristik),0.285,0.694,0.318,0.172,1.720,NaN,NaN,NaN
acak (seed 0),0.185,0.501,0.188,0.101,1.015,NaN,NaN,NaN


In [5]:
from sklearn.metrics import confusion_matrix
for name, s in scores.items():
    tn, fp, fn, tp = confusion_matrix(y_te, (s >= 0.5).astype(int)).ravel()
    print(f"{name:14s} threshold 0,5 -> TP={tp} FP={fp} FN={fn} TN={tn}")

logreg         threshold 0,5 -> TP=168 FP=241 FN=3902 TN=17675
random_forest  threshold 0,5 -> TP=9 FP=28 FN=4061 TN=17888


## Pencocokan dengan `results/metrics.csv`

Selisih harus nol atau sangat kecil (toleransi numerik antar-environment).

In [6]:
ref = pd.read_csv(ROOT / "results" / "metrics.csv")
for name in ["logreg", "random_forest"]:
    r = ref[(ref.scenario == "baseline_thr0.5") & (ref.model == name) & (ref.seed == 0)].iloc[0]
    for col in ["pr_auc", "roc_auc", "precision_at_10pct", "recall_at_10pct", "brier", "ece", "precision", "recall"]:
        assert abs(rows[name][col] - r[col]) < 1e-6, (name, col, rows[name][col], r[col])
print("Cocok dengan results/metrics.csv (baseline_thr0.5, seed 0).")

Cocok dengan results/metrics.csv (baseline_thr0.5, seed 0).
